# 04e — Feed completeness: what the data can and cannot show

Reviewer 1: "the differences observed between countries may partly reflect data availability and feed
harmonisation rather than actual railway connectivity. Although this issue is acknowledged briefly, its effect is
not quantified." This notebook quantifies it (register REP-4, CON-10, VAL-4). Code: `traintranv2/nb04_feeds.py`.

| § | step | output (`output/kosice/explore/`) |
|---|---|---|
| 1 | the feeds: size, time coverage, exceptions, known omissions | `feed_table.csv` |
| 2 | what each national feed adds over the Slovak feed alone | `feed_effect.csv` |
| 3 | the station-merge radius (VAL-4) | `dedup_radius.csv`, `dedup_changes.csv` |

**How the effect is measured.** For each destination country, the one-transfer analysis of 03b is run twice on the
same window: with the Slovak feed alone (every train from ZSR_SK), and with ZSR_SK plus the country's own feed (the
headline). The difference — LAUs and people reached only because the national feed exists — is the part of a
country's figure that depends on data availability. A country whose national feed is incomplete (Ukraine) shows
what is missing as a small difference that cannot be told apart from an absence of service: the result is
**observed connectivity within the available feeds** (CON-10).

In [1]:
import os, sys

# Run from the project root so every relative path resolves the same from notebooks/ or the root.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

Working directory: E:\notes\api_zssk\project


In [2]:
import pandas as pd
import traintranv2  # noqa: F401  (PROJ data fix)
from traintranv2.config import default_config
from traintranv2 import nb02_connections as C
from traintranv2 import nb03_foreign as F
from traintranv2 import nb04_feeds as FD
from traintranv2.nb01a_feeds import write_meta
from traintranv2.nb01b_stations import load_lau_subset

cfg = default_config()
OUT = os.path.join(cfg.out_dir, 'explore')
tags = pd.read_csv(os.path.join(cfg.out_dir, 'station_tags.csv'), dtype=str)
L = C.lau_with_city_units(load_lau_subset(cfg), cfg)
tags = C.tags_with_city_units(tags, cfg, L)
hubs = C.hub_groups(cfg, tags)
pop, _ = F.lau_population(L)

## 1. The feeds

Size, the calendar each feed declares and the analysis window used (01a), and its exceptions (`calendar_dates`
rows adding or removing service days: holidays, engineering works, seasonal days). Known omissions, from the
feeds' documentation and the earlier notebooks:

| feed | known limitation |
|---|---|
| ZSR_SK | complete for ŽSR / ZSSK services, incl. international trains on Slovak track; the reference feed |
| DRAHY_CZ | whole year; carries the Czech-operated trains abroad too |
| MAV_HU | older snapshot (Jan–Nov 2026, 301 days) |
| BAHN_AT | trains published in pieces at region and border points (joined in 02, `join_train_pieces`); window Mar–Dec 2026 |
| PL_IC | PKP Intercity only, 101 days (May–Aug 2026) |
| PL_REG | all other Polish carriers; a rolling feed of 32 days (June–July 2026) |
| UKRZ_UA | a subset of Ukrzaliznytsia's services (international and western Ukraine) |

In [3]:
ft = FD.feed_table(cfg, tags)
display(ft)

,feed_key,file,bytes,agencies,routes,trips,stops,stations,route_types,calendar_dates_added,calendar_dates_removed,service_start,service_end,window_start,window_end,window_days,role
0,ZSR_SK,sk_zsr.gtfs.zip,271560,6,2309,2512,887,887,100;102;103;105;106,1450,3660,2025-12-14,2026-12-12,2025-12-14,2026-12-12,364,domestic
1,MAV_HU,hu_mav.gtfs.zip,2319938,3,12317,37650,1197,1134,2;3,0,0,2026-01-19,2026-11-15,2026-01-19,2026-11-15,301,foreign
2,UKRZ_UA,ua-ukrzaliznytsya.gtfs.zip,3491128,4,79,79,118,116,102;106,23479,0,2025-12-14,2026-12-12,2025-12-14,2026-12-12,364,foreign
3,BAHN_AT,BAHN_AT.zip,5197760,13,22960,46423,7683,1135,2;3,10106,492655,2025-12-14,2026-12-12,2026-03-12,2026-12-12,276,foreign
4,DRAHY_CZ,cz_CZPTT_new.gtfs.zip,50175022,42,8801,77997,9511,2729,100;102;103;105;106;714,1194582,0,2025-12-14,2026-12-12,2025-12-14,2026-12-12,364,foreign
5,PL_IC,PL_kasznia.zip,109649923,1,9,58101,2098,498,2;3,0,0,2026-05-18,2026-08-26,2026-05-18,2026-08-26,101,foreign
6,PL_REG,polish-trains.zip,10063938,13,194,35516,12187,3115,101;102;103;106;109;3,6763,0,2026-06-04,2026-07-06,2026-06-05,2026-07-06,32,foreign


## 2. What each national feed adds

Per country, inbound and outbound: LAUs and population reached directly or with one transfer, with ZSR_SK alone and
with the national feed added. Same window, same rules (03b).

In [4]:
fe = FD.feed_effect(cfg, tags, hubs, pop)
assert (fe['added_lau'] >= 0).all(), 'adding a feed cannot lose a LAU'
show = fe[fe.direction == 'inbound'].pivot_table(index='country', columns='variant', values=['n_lau', 'population'])
show[('share of population from the national feed', '')] = (
    1 - show[('population', 'ZSR_SK alone')] / show[('population', 'ZSR_SK + national feed')]).round(3)
display(show)

n_lau                          population  \
variant ZSR_SK + national feed ZSR_SK alone ZSR_SK + national feed   
country                                                              
AT                        31.0         15.0              3160473.0   
CZ                       535.0         36.0              6068419.0   
HU                       186.0         13.0              3844552.0   
PL                        21.0         11.0              4193156.0   
UA                         3.0          3.0               240896.0   

                     share of population from the national feed  
variant ZSR_SK alone                                             
country                                                          
AT         2714901.0                                      0.141  
CZ         2824225.0                                      0.535  
HU         1907064.0                                      0.504  
PL         2728711.0                                      0.349  
UA          240896.0                                      0.000

## 3. The station-merge radius (VAL-4)

For the feeds without GTFS parent stations (ZSR_SK, MÁV, UZ), stops of the same name are one station if they lie
within 2 km (01b, the domestic run's rule). The same at 0.5, 1 and 5 km: how many stations, and which merges change.
(Feeds with parent stations — CZ, AT, PL — group platforms by the parent and are not affected.)

In [5]:
dr, dc = FD.dedup_radius(cfg)
display(dr.pivot_table(index='feed_key', columns='radius_km', values='stations'))
print(f'stations that a radius other than 2 km would form differently: {len(dc)}')
if len(dc):
    display(dc.groupby(['feed_key', 'radius_km']).size().rename('changed').to_frame().T)
    display(dc.head(15))

radius_km,0.5,1.0,2.0,5.0
feed_key,,,,
MAV_HU,1197.0,1197.0,1197.0,1197.0
UKRZ_UA,116.0,116.0,116.0,116.0
ZSR_SK,887.0,887.0,887.0,887.0


stations that a radius other than 2 km would form differently: 0


## 4. Outputs

In [6]:
for name, key, frame in [('feed_table.csv', 'explore_feed_table', ft), ('feed_effect.csv', 'explore_feed_effect', fe),
                         ('dedup_radius.csv', 'explore_dedup_radius', dr), ('dedup_changes.csv', 'explore_dedup_changes', dc)]:
    p = os.path.join(OUT, name)
    frame.to_csv(p, index=False, encoding='utf-8')
    write_meta(p, key, frame, {'Runs': 'headline country runs (03)'})
    print(f'[wrote] {p} ({len(frame):,} rows) + .meta.md')

[wrote] output\kosice\explore\feed_table.csv (7 rows) + .meta.md
[wrote] output\kosice\explore\feed_effect.csv (20 rows) + .meta.md
[wrote] output\kosice\explore\dedup_radius.csv (12 rows) + .meta.md
[wrote] output\kosice\explore\dedup_changes.csv (0 rows) + .meta.md
